In [1]:
import sys, os

dir = os.path.abspath('..')
sys.path.append(os.path.join(dir))
while not os.path.basename(dir) == 'v2':
    parent = os.path.dirname(dir)
    if parent == dir:
        raise FileNotFoundError("No parent directory named 'v2' found.")
    dir = parent

# CHECK NISHANTH

## Add edge

In [5]:
ORIGIN_NAME = 'check-Nishant'
SYNTH_NAME = 'check-Nishant-synth-feedback-add-edge'

In [7]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/5 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 2115
Duration of Video: 70.5 seconds


Importing:  20%|██████▊                           | 1/5 [00:04<00:17,  4.34s/it]

Total # of frames saved: 71
FPS: 30.0
Samplerate: 1
Total # of frames: 2383
Duration of Video: 79.43333333333334 seconds


Importing:  40%|█████████████▌                    | 2/5 [00:09<00:13,  4.57s/it]

...

Total # of frames saved: 28


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 2515
Duration of Video: 41.916666666666664 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:03<00:00,  3.09s/it]

Total # of frames saved: 42


In [8]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After passing to teammate, wait until teammate passes the ball back to you, once you have a ball again, shoot for a goal"
code = '''behavior CoachBehavior():
    do Speak("I will check to an open space at an angle to receive a pass.")
    do MoveTo(λ_target0())
    do Speak("Now I will wait for my teammate to make the pass.")
    do Idle() until λ_termination0(simulation(), None)
    do Speak("I see the pass coming, I will move to get possession of the ball.")
    do GetBallPossession(ball)
    if λ_precondition0(simulation(), None):
        do Speak("The opponent is pressuring me, so I will pass the ball back to my teammate.")
        do Pass(teammate)
    else:
        do Speak("I have enough space, so I will dribble forward and take a shot on goal.")
        do Shoot(goal)

A1target_0 = AtAngle({'player': 'Coach', 'ball': 'ball', 'left': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}, 'right': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_0 = CloseTo({'obj': 'opponent', 'ref': 'Coach', 'max': {'avg': 2.1, 'std': 0.6}})

def λ_target0():
	return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
	return A1termination_0.bool(simulation())

def λ_precondition0(scene, sample):
	return A1precondition_0.bool(simulation())
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I will check to an open space at an angle to receive a pass.")
    do MoveTo(λ_target0())
    do Speak("Now I will wait for my teammate to make the pass.")
    do Idle() until λ_termination0(simulation(), None)
    do Speak("I see the pass coming, I will move to get possession of the ball.")
    do GetBallPossession(ball)
    if λ_precondition0(simulation(), None):
        do Speak("The opponent is pressuring me, so I will pass the ball back to my teammate.")
        do Pass(teammate)
    else:
        do Speak("I have enough space, so I will dribble forward and take a shot on goal.")
        do Shoot(goal)

A1target_0 = AtAngle({'player': 'Coach', 'ball': 'ball', 'left': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}, 'right': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_0 = CloseTo({'obj': 'opponent', 'ref': 'Coach', 'max': {'avg': 2.1, 'std': 0.6}})

def λ_target0():
	return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
	return A1termination_0.bool(simulation())

def λ_precondition0(scene, sample):
	return A1precondition_0.bool(simulation())




# Parameters for variance
coach_start_dist = Uniform(5, 8)  # initial distance from teammate
coach_check_dist = Uniform(4, 6)   # how much closer coach checks
coach_check_angle = Uniform(-45, 45)  # angle of check (degrees)
opponent_dist = Uniform(1, 5)         # distance behind coach
opponent_speed = Uniform(5, 7)        # opponent's movement speed

# Behaviors
behavior TeammatePass():
    do Idle() for 1.0 seconds  # Give coach time to start 
    do GetBallPossession(ball)
    print("got ball")
    do Idle() for 5.0 seconds
    do Pass(ego)
    do Idle()

behavior OpponentFollowCoach():
    do Idle() for 0.5 seconds  # Wait for coach to start checking
    speed = float(opponent_speed)
    do SetPlayerSpeed(speed)
    while True:
        do MoveToBehavior(ego.position)
        do Idle() for 0.1 seconds

# Place teammate (AI) at origin
teammate = new Player at (0, 0, 0), with name "teammate", with team "green", with behavior TeammatePass()

# Place coach (human) in front of teammate
ego = new Coach ahead of teammate by coach_start_dist, with name "Coach", with team "blue", with behavior CoachBehavior()

# Place opponent ahead of coach (further from goal than coach)
opponent = new Player ahead of ego by opponent_dist, facing toward ego, with name "opponent", with team "red", with behavior OpponentFollowCoach()

# Ball at teammate's feet
ball = new Ball ahead of teammate by 0.5

goal = new Goal at (0, 17, 0)
'''

In [9]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code, response = fixer.run()

In [10]:
response

'Based on the coach\'s feedback, I will make the following corrections to the Scenic program.\n\n### Summary of Feedback and Changes\n\n*   **Feedback:** The coach pointed out that when the avatar is under pressure and passes the ball back to the teammate, its behavior incorrectly terminates. The correct behavior should be to wait for the teammate to pass the ball back, and once the avatar receives it again, it should take a shot on goal.\n\n*   **Changes:** I modified the `if` block within the `CoachBehavior()` function, which handles the scenario where the opponent is close. After the existing `do Pass(teammate)` action, I added the following sequence:\n    1.  A `Speak` action to narrate the intent to wait for a return pass.\n    2.  A `do ReceiveBall()` action, which makes the coach wait until regaining possession of the ball.\n    3.  Another `Speak` action to announce the final shot.\n    4.  A `do Shoot(goal)` action to perform the shot on goal as requested.\n\nThis change ensur

In [11]:
fixed_code

'python\nbehavior CoachBehavior():\n    do Speak("I will check to an open space at an angle to receive a pass.")\n    do MoveTo(λ_target0())\n    do Speak("Now I will wait for my teammate to make the pass.")\n    do Idle() until λ_termination0(simulation(), None)\n    do Speak("I see the pass coming, I will move to get possession of the ball.")\n    do GetBallPossession(ball)\n    if λ_precondition0(simulation(), None):\n        do Speak("The opponent is pressuring me, so I will pass the ball back to my teammate.")\n        do Pass(teammate)\n        # CHANGE: Added actions to wait for a return pass and then shoot, as per feedback.\n        # The original behavior ended after passing, which was incorrect.\n        do Speak("I\'ll wait for the return pass.")\n        do ReceiveBall()\n        do Speak("Now I have the ball again, I\'ll take a shot.")\n        do Shoot(goal)\n    else:\n        do Speak("I have enough space, so I will dribble forward and take a shot on goal.")\n        do

## Add node

In [18]:
ORIGIN_NAME = 'check-Nishant'
SYNTH_NAME = 'check-Nishant-synth-feedback-add-node'

In [20]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/5 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 2115
Duration of Video: 70.5 seconds


Importing:  20%|██████▊                           | 1/5 [00:04<00:17,  4.30s/it]

Total # of frames saved: 71
FPS: 30.0
Samplerate: 1
Total # of frames: 2383
Duration of Video: 79.43333333333334 seconds


Importing:  40%|█████████████▌                    | 2/5 [00:09<00:13,  4.64s/it]

...

Total # of frames saved: 28


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 3108
Duration of Video: 51.8 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:03<00:00,  3.93s/it]

Total # of frames saved: 52


In [21]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After you are in position and created an angle of pass you can receive the ball from the teammate, but it’s also possible that the teammate will decide to go for the goal, if the opponent decides to pressure you. In this case keep the opponent away from the goal."
code = '''behavior CoachBehavior():
    do Speak("I will check to an open space at an angle to receive a pass.")
    do MoveTo(λ_target0())
    do Speak("Now I will wait for my teammate to make the pass.")
    do Idle() until λ_termination0(simulation(), None)
    do Speak("I see the pass coming, I will move to get possession of the ball.")
    do GetBallPossession(ball)
    if λ_precondition0(simulation(), None):
        do Speak("The opponent is pressuring me, so I will pass the ball back to my teammate.")
        do Pass(teammate)
    else:
        do Speak("I have enough space, so I will dribble forward and take a shot on goal.")
        do Shoot(goal)

A1target_0 = AtAngle({'player': 'Coach', 'ball': 'ball', 'left': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}, 'right': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_0 = CloseTo({'obj': 'opponent', 'ref': 'Coach', 'max': {'avg': 2.1, 'std': 0.6}})

def λ_target0():
	return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
	return A1termination_0.bool(simulation())

def λ_precondition0(scene, sample):
	return A1precondition_0.bool(simulation())
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I will check to an open space at an angle to receive a pass.")
    do MoveTo(λ_target0())
    do Speak("Now I will wait for my teammate to make the pass.")
    do Idle() until λ_termination0(simulation(), None)
    do Speak("I see the pass coming, I will move to get possession of the ball.")
    do GetBallPossession(ball)
    if λ_precondition0(simulation(), None):
        do Speak("The opponent is pressuring me, so I will pass the ball back to my teammate.")
        do Pass(teammate)
    else:
        do Speak("I have enough space, so I will dribble forward and take a shot on goal.")
        do Shoot(goal)

A1target_0 = AtAngle({'player': 'Coach', 'ball': 'ball', 'left': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}, 'right': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_0 = CloseTo({'obj': 'opponent', 'ref': 'Coach', 'max': {'avg': 2.1, 'std': 0.6}})

def λ_target0():
	return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
	return A1termination_0.bool(simulation())

def λ_precondition0(scene, sample):
	return A1precondition_0.bool(simulation())




# Parameters for variance
coach_start_dist = Uniform(5, 8)  # initial distance from teammate
coach_check_dist = Uniform(4, 6)   # how much closer coach checks
coach_check_angle = Uniform(-45, 45)  # angle of check (degrees)
opponent_dist = Uniform(1, 5)         # distance behind coach
opponent_speed = Uniform(5, 7)        # opponent's movement speed

# Behaviors
behavior TeammatePass():
    do Idle() for 1.0 seconds  # Give coach time to start 
    do GetBallPossession(ball)
    print("got ball")
    do Idle() for 5.0 seconds
    do Pass(ego)
    do Idle()

behavior OpponentFollowCoach():
    do Idle() for 0.5 seconds  # Wait for coach to start checking
    speed = float(opponent_speed)
    do SetPlayerSpeed(speed)
    while True:
        do MoveToBehavior(ego.position)
        do Idle() for 0.1 seconds

# Place teammate (AI) at origin
teammate = new Player at (0, 0, 0), with name "teammate", with team "green", with behavior TeammatePass()

# Place coach (human) in front of teammate
ego = new Coach ahead of teammate by coach_start_dist, with name "Coach", with team "blue", with behavior CoachBehavior()

# Place opponent ahead of coach (further from goal than coach)
opponent = new Player ahead of ego by opponent_dist, facing toward ego, with name "opponent", with team "red", with behavior OpponentFollowCoach()

# Ball at teammate's feet
ball = new Ball ahead of teammate by 0.5

goal = new Goal at (0, 17, 0)
'''

In [22]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_check_add_node, response_check_add_node = fixer.run()

In [23]:
response_check_add_node

'### Fix Explanation\n\n*   **What the feedback was:** The coach explained that after moving to an open space, there are two possibilities. The first is that the teammate passes the ball to the coach (the original behavior). The second, new possibility is that if the opponent pressures the coach, the teammate might decide to attack the goal directly. In this case, the coach\'s new job is to "keep the opponent away from the goal" to help the teammate, which means drawing their defender away from the play.\n\n*   **What I changed in the Scenic code:**\n    1.  I introduced a new condition, `A1precondition_1`, which uses the `Pressure` API to check if the opponent is actively pressuring the coach. I also added a corresponding lambda function `λ_precondition1`.\n    2.  In `CoachBehavior`, I modified the `do Idle()` state so that it terminates when *either* the teammate passes (`λ_termination0`) *or* the opponent applies pressure (`λ_precondition1`).\n    3.  I added an `if/else` block to 

In [24]:
fixed_code_check_add_node

'python\nbehavior CoachBehavior():\n    do Speak("I will check to an open space at an angle to receive a pass.")\n    do MoveTo(λ_target0())\n    do Speak("Now I will wait for my teammate to make the pass.")\n    \n    # MODIFICATION: Changed the termination condition for the Idle state.\n    # The coach now waits for EITHER the teammate to pass (λ_termination0) OR for the opponent to apply pressure (λ_precondition1).\n    # This allows the behavior to react to the opponent\'s actions, as per the feedback.\n    do Idle() until λ_termination0(simulation(), None) or λ_precondition1(simulation(), None)\n\n    # MODIFICATION: Added a conditional branch to handle the two possible outcomes.\n    # The feedback states that if the opponent pressures the coach, the teammate might go for goal.\n    # In this case, the coach\'s role changes to drawing the opponent away.\n    if λ_precondition1(simulation(), None):\n        # NEW BEHAVIOR: If pressured, the coach holds their position. Since the op

## Delete node

In [43]:
ORIGIN_NAME = 'check-Nishant'
SYNTH_NAME = 'check-Nishant-synth-feedback-delete-node'

In [27]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/5 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 2115
Duration of Video: 70.5 seconds


Importing:  20%|██████▊                           | 1/5 [00:04<00:16,  4.23s/it]

Total # of frames saved: 71
FPS: 30.0
Samplerate: 1
Total # of frames: 2383
Duration of Video: 79.43333333333334 seconds


Importing:  40%|█████████████▌                    | 2/5 [00:08<00:13,  4.51s/it]

...

Total # of frames saved: 28


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 2634
Duration of Video: 43.9 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:03<00:00,  3.09s/it]

Total # of frames saved: 44


In [28]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After receiving the ball we want to always pass back to the teammate. Don’t go for a goal even if opponent is not pressuring you"
code = '''behavior CoachBehavior():
    do Speak("I will check to an open space at an angle to receive a pass.")
    do MoveTo(λ_target0())
    do Speak("Now I will wait for my teammate to make the pass.")
    do Idle() until λ_termination0(simulation(), None)
    do Speak("I see the pass coming, I will move to get possession of the ball.")
    do GetBallPossession(ball)
    if λ_precondition0(simulation(), None):
        do Speak("The opponent is pressuring me, so I will pass the ball back to my teammate.")
        do Pass(teammate)
    else:
        do Speak("I have enough space, so I will dribble forward and take a shot on goal.")
        do Shoot(goal)

A1target_0 = AtAngle({'player': 'Coach', 'ball': 'ball', 'left': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}, 'right': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_0 = CloseTo({'obj': 'opponent', 'ref': 'Coach', 'max': {'avg': 2.1, 'std': 0.6}})

def λ_target0():
	return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
	return A1termination_0.bool(simulation())

def λ_precondition0(scene, sample):
	return A1precondition_0.bool(simulation())
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I will check to an open space at an angle to receive a pass.")
    do MoveTo(λ_target0())
    do Speak("Now I will wait for my teammate to make the pass.")
    do Idle() until λ_termination0(simulation(), None)
    do Speak("I see the pass coming, I will move to get possession of the ball.")
    do GetBallPossession(ball)
    if λ_precondition0(simulation(), None):
        do Speak("The opponent is pressuring me, so I will pass the ball back to my teammate.")
        do Pass(teammate)
    else:
        do Speak("I have enough space, so I will dribble forward and take a shot on goal.")
        do Shoot(goal)

A1target_0 = AtAngle({'player': 'Coach', 'ball': 'ball', 'left': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}, 'right': {'theta': {'avg': 40.32, 'std': 6.4}, 'dist': {'avg': 5.8, 'std': 0.8}}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_0 = CloseTo({'obj': 'opponent', 'ref': 'Coach', 'max': {'avg': 2.1, 'std': 0.6}})

def λ_target0():
	return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
	return A1termination_0.bool(simulation())

def λ_precondition0(scene, sample):
	return A1precondition_0.bool(simulation())




# Parameters for variance
coach_start_dist = Uniform(5, 8)  # initial distance from teammate
coach_check_dist = Uniform(4, 6)   # how much closer coach checks
coach_check_angle = Uniform(-45, 45)  # angle of check (degrees)
opponent_dist = Uniform(1, 5)         # distance behind coach
opponent_speed = Uniform(5, 7)        # opponent's movement speed

# Behaviors
behavior TeammatePass():
    do Idle() for 1.0 seconds  # Give coach time to start 
    do GetBallPossession(ball)
    print("got ball")
    do Idle() for 5.0 seconds
    do Pass(ego)
    do Idle()

behavior OpponentFollowCoach():
    do Idle() for 0.5 seconds  # Wait for coach to start checking
    speed = float(opponent_speed)
    do SetPlayerSpeed(speed)
    while True:
        do MoveToBehavior(ego.position)
        do Idle() for 0.1 seconds

# Place teammate (AI) at origin
teammate = new Player at (0, 0, 0), with name "teammate", with team "green", with behavior TeammatePass()

# Place coach (human) in front of teammate
ego = new Coach ahead of teammate by coach_start_dist, with name "Coach", with team "blue", with behavior CoachBehavior()

# Place opponent ahead of coach (further from goal than coach)
opponent = new Player ahead of ego by opponent_dist, facing toward ego, with name "opponent", with team "red", with behavior OpponentFollowCoach()

# Ball at teammate's feet
ball = new Ball ahead of teammate by 0.5

goal = new Goal at (0, 17, 0)
'''

In [29]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_check_delete_node, response_check_delete_node = fixer.run()

In [30]:
response_check_delete_node

'### Fix Explanation\n\nThe coach\'s feedback was to change the program so that after the coach avatar receives the ball, it *always* passes the ball back to the teammate. The original program had a conditional behavior where the coach would sometimes shoot on the goal if the opponent was not applying pressure.\n\nTo implement this feedback, I made the following changes to the `CoachBehavior()` function:\n- I removed the entire `if/else` block that came after the `do GetBallPossession(ball)` action.\n- This conditional block was responsible for deciding whether to pass to the teammate or shoot at the goal based on the opponent\'s proximity.\n- I replaced it with an unconditional sequence of actions that first narrates the intention to pass and then executes the `Pass(teammate)` action, ensuring this is the only possible outcome after getting the ball.\n\nThis change directly addresses the coach\'s request, simplifying the logic to a fixed sequence of actions as instructed.\n\n### Fixed

In [31]:
fixed_code_check_delete_node

'python\nbehavior CoachBehavior():\n    do Speak("I will check to an open space at an angle to receive a pass.")\n    do MoveTo(λ_target0())\n    do Speak("Now I will wait for my teammate to make the pass.")\n    do Idle() until λ_termination0(simulation(), None)\n    do Speak("I see the pass coming, I will move to get possession of the ball.")\n    do GetBallPossession(ball)\n    # MODIFICATION: Based on the feedback, the coach should always pass the ball back to the teammate\n    # after receiving it. The original conditional logic (if/else) which allowed for\n    # shooting on the goal has been removed.\n    do Speak("I will pass the ball back to my teammate.")\n    do Pass(teammate)\n\nA1target_0 = AtAngle({\'player\': \'Coach\', \'ball\': \'ball\', \'left\': {\'theta\': {\'avg\': 40.32, \'std\': 6.4}, \'dist\': {\'avg\': 5.8, \'std\': 0.8}}, \'right\': {\'theta\': {\'avg\': 40.32, \'std\': 6.4}, \'dist\': {\'avg\': 5.8, \'std\': 0.8}}})\nA1termination_0 = MakePass({\'player\': \'t

# Overlap Nishant

## Add edge

In [46]:
ORIGIN_NAME = 'overlap-Nishant'
SYNTH_NAME = 'overlap-Nishant-synth-feedback-add-edge'

In [48]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/6 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 1299
Duration of Video: 43.3 seconds


Importing:  17%|█████▋                            | 1/6 [00:02<00:13,  2.65s/it]

Total # of frames saved: 44
FPS: 30.0
Samplerate: 1
Total # of frames: 962
Duration of Video: 32.06666666666667 seconds


Importing:  33%|███████████▎                      | 2/6 [00:04<00:09,  2.27s/it]

...

Total # of frames saved: 21


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 1731
Duration of Video: 28.85 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:02<00:00,  2.15s/it]

Total # of frames saved: 29


In [49]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "Once you close to goal pass the ball to teammate"
code = '''behavior CoachBehavior():
    do Speak("I will make an overlapping run to create space and attract the defender.")
    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)
    do Speak("The teammate is passing. I will receive the ball.")
    do ReceiveBall()
    if λ_precondition1(simulation(), None):
        do Speak("I have enough space, so I will dribble towards the goal.")
        do MoveTo(goal)
    else:
        do Speak("The defender is too close, so I will pass the ball back to my teammate.")
        do Pass(teammate)

A1target_0 = Overlap({'player': 'Coach', 'ball': 'ball', 'goal': 'goal', 'opponent': 'defender1', 'theta': {'avg': 37.49500473945939, 'std': 4.316867385966453}, 'dist': {'avg': 4.144414138072041, 'std': 1.1398863177626926}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_1 = DistanceTo({'to': 'defender1', 'from': 'Coach', 'operator': 'greater_than', 'min': {'avg': 4.238624128913989, 'std': 0.613329061405903}})
A2precondition_1 = HasBallPossession({'player': 'Coach'})

def λ_target0():
    return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
    return A1termination_0.bool(simulation())

def λ_precondition1(scene, sample):
    return A1precondition_1.bool(simulation()) and A2precondition_1.bool(simulation())
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I will make an overlapping run to create space and attract the defender.")
    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)
    do Speak("The teammate is passing. I will receive the ball.")
    do ReceiveBall()
    if λ_precondition1(simulation(), None):
        do Speak("I have enough space, so I will dribble towards the goal.")
        do MoveTo(goal)
    else:
        do Speak("The defender is too close, so I will pass the ball back to my teammate.")
        do Pass(teammate)

A1target_0 = Overlap({'player': 'Coach', 'ball': 'ball', 'goal': 'goal', 'opponent': 'defender1', 'theta': {'avg': 37.49500473945939, 'std': 4.316867385966453}, 'dist': {'avg': 4.144414138072041, 'std': 1.1398863177626926}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_1 = DistanceTo({'to': 'defender1', 'from': 'Coach', 'operator': 'greater_than', 'min': {'avg': 4.238624128913989, 'std': 0.613329061405903}})
A2precondition_1 = HasBallPossession({'player': 'Coach'})

def λ_target0():
    return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
    return A1termination_0.bool(simulation())

def λ_precondition1(scene, sample):
    return A1precondition_1.bool(simulation()) and A2precondition_1.bool(simulation())




opponent_y_distance = Uniform(3, 5)
opponent_x_distance = Uniform(-2, 2)
ego_x_distance = Uniform(-2, 2)
ego_y_distance = Uniform(-1, -2)

# Ensure teammate and opponent are on the same side
#require (opponent_x_distance < 0 and ego_x_distance < 0) or (opponent_x_distance >= 0 and ego_x_distance >= 0)

behavior Follow(obj):
    while ego.position.y > 1:
        do MoveToBehavior(obj, distance = 2, status = f"Follow {obj.name}")

behavior TeammateBehavior():
    do Idle() for 1 seconds
    do GetBallPossession(ball)
    do Idle() until ego.position.y > 2
    print("ego at good position")
    do Idle() for 1 seconds
    do Pass(ego, slow=False) until (distance from opponent to ego) <= 3
    print("pass happened")
    do DribbleTo(goal) until (distance from opponent to ego) > 3
    do Idle() for 2 seconds
    

behavior DefenderBehavior():
    do Idle() for 1 seconds
    do Idle() until ego.position.y > 1
    do Follow(ego) until ego.gameObject.ballPossession
    

teammate = new Player at (0, 0, 0),
      with behavior TeammateBehavior(), with name "teammate"

ball = new Ball ahead of teammate by 1

ego = new Coach at (ego_x_distance, ego_y_distance, 0), with name "Coach", with team "blue", with behavior CoachBehavior()

opponent = new Player at (0, Uniform(4, 6), 0), with name "defender1",
            with behavior DefenderBehavior()

goal = new Goal at (0, 17, 0)

'''

In [50]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_overlap_synth_add_edge, response_overlap_synth_add_edge = fixer.run()

In [51]:
response_overlap_synth_add_edge

'Based on the coach\'s feedback, I will adjust the `CoachBehavior` to ensure the coach passes the ball to the teammate when near the goal.\n\n- **Feedback Analysis**: The coach stated, "...once he is near the goal, he should pass to the teammate." The current program has the coach move towards the goal (`do MoveTo(goal)`) but never performs a subsequent action. This move action lacks a termination condition.\n\n- **Code Changes**:\n  1. I introduced a new termination condition, `λ_termination1`, which uses the `DistanceTo` constraint API. This condition checks if the coach is less than 5 units away from the goal.\n  2. I modified the `MoveTo(goal)` action within the `if` block to use this new termination condition: `do MoveTo(goal) until λ_termination1(...)`.\n  3. After the `MoveTo` action, I added a `Speak` action to narrate the pass and then the `Pass(teammate)` action to execute the pass, as requested by the coach.\n\nThis ensures that if the coach has space to dribble, they will m

In [52]:
fixed_code_overlap_synth_add_edge

'python\nbehavior CoachBehavior():\n    do Speak("I will make an overlapping run to create space and attract the defender.")\n    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)\n    do Speak("The teammate is passing. I will receive the ball.")\n    do ReceiveBall()\n    if λ_precondition1(simulation(), None):\n        do Speak("I have enough space, so I will dribble towards the goal.")\n        # CHANGE: Added a termination condition to stop the coach\'s movement when they get close to the goal.\n        # This addresses the feedback that the coach should pass when near the goal.\n        do MoveTo(goal) until λ_termination1(simulation(), None)\n        # ADD: Added a narration for the new pass action.\n        do Speak("I am near the goal, so I will pass to my teammate.")\n        # ADD: Added the Pass action to implement the coach\'s feedback.\n        do Pass(teammate)\n    else:\n        do Speak("The defender is too close, so I will pass the ball back to my teamma

## Add node

In [54]:
ORIGIN_NAME = 'overlap-Nishant'
SYNTH_NAME = 'overlap-Nishant-synth-feedback-add-node'

In [55]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/6 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 1299
Duration of Video: 43.3 seconds


Importing:  17%|█████▋                            | 1/6 [00:02<00:12,  2.56s/it]

Total # of frames saved: 44
FPS: 30.0
Samplerate: 1
Total # of frames: 962
Duration of Video: 32.06666666666667 seconds


Importing:  33%|███████████▎                      | 2/6 [00:04<00:08,  2.21s/it]

...

Total # of frames saved: 21


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 2222
Duration of Video: 37.03333333333333 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:02<00:00,  2.82s/it]

Total # of frames saved: 38


In [56]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After you’ve overlapped the teammate, you might not receive the ball if the teammate decides to go for the goal, in this case create an angle of pass near the goal so you are in a good supporting position for the teammate and ready to receive a pass.  "
code = '''behavior CoachBehavior():
    do Speak("I will make an overlapping run to create space and attract the defender.")
    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)
    do Speak("The teammate is passing. I will receive the ball.")
    do ReceiveBall()
    if λ_precondition1(simulation(), None):
        do Speak("I have enough space, so I will dribble towards the goal.")
        do MoveTo(goal)
    else:
        do Speak("The defender is too close, so I will pass the ball back to my teammate.")
        do Pass(teammate)

A1target_0 = Overlap({'player': 'Coach', 'ball': 'ball', 'goal': 'goal', 'opponent': 'defender1', 'theta': {'avg': 37.49500473945939, 'std': 4.316867385966453}, 'dist': {'avg': 4.144414138072041, 'std': 1.1398863177626926}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_1 = DistanceTo({'to': 'defender1', 'from': 'Coach', 'operator': 'greater_than', 'min': {'avg': 4.238624128913989, 'std': 0.613329061405903}})
A2precondition_1 = HasBallPossession({'player': 'Coach'})

def λ_target0():
    return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
    return A1termination_0.bool(simulation())

def λ_precondition1(scene, sample):
    return A1precondition_1.bool(simulation()) and A2precondition_1.bool(simulation())
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I will make an overlapping run to create space and attract the defender.")
    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)
    do Speak("The teammate is passing. I will receive the ball.")
    do ReceiveBall()
    if λ_precondition1(simulation(), None):
        do Speak("I have enough space, so I will dribble towards the goal.")
        do MoveTo(goal)
    else:
        do Speak("The defender is too close, so I will pass the ball back to my teammate.")
        do Pass(teammate)

A1target_0 = Overlap({'player': 'Coach', 'ball': 'ball', 'goal': 'goal', 'opponent': 'defender1', 'theta': {'avg': 37.49500473945939, 'std': 4.316867385966453}, 'dist': {'avg': 4.144414138072041, 'std': 1.1398863177626926}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_1 = DistanceTo({'to': 'defender1', 'from': 'Coach', 'operator': 'greater_than', 'min': {'avg': 4.238624128913989, 'std': 0.613329061405903}})
A2precondition_1 = HasBallPossession({'player': 'Coach'})

def λ_target0():
    return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
    return A1termination_0.bool(simulation())

def λ_precondition1(scene, sample):
    return A1precondition_1.bool(simulation()) and A2precondition_1.bool(simulation())




opponent_y_distance = Uniform(3, 5)
opponent_x_distance = Uniform(-2, 2)
ego_x_distance = Uniform(-2, 2)
ego_y_distance = Uniform(-1, -2)

# Ensure teammate and opponent are on the same side
#require (opponent_x_distance < 0 and ego_x_distance < 0) or (opponent_x_distance >= 0 and ego_x_distance >= 0)

behavior Follow(obj):
    while ego.position.y > 1:
        do MoveToBehavior(obj, distance = 2, status = f"Follow {obj.name}")

behavior TeammateBehavior():
    do Idle() for 1 seconds
    do GetBallPossession(ball)
    do Idle() until ego.position.y > 2
    print("ego at good position")
    do Idle() for 1 seconds
    do Pass(ego, slow=False) until (distance from opponent to ego) <= 3
    print("pass happened")
    do DribbleTo(goal) until (distance from opponent to ego) > 3
    do Idle() for 2 seconds
    

behavior DefenderBehavior():
    do Idle() for 1 seconds
    do Idle() until ego.position.y > 1
    do Follow(ego) until ego.gameObject.ballPossession
    

teammate = new Player at (0, 0, 0),
      with behavior TeammateBehavior(), with name "teammate"

ball = new Ball ahead of teammate by 1

ego = new Coach at (ego_x_distance, ego_y_distance, 0), with name "Coach", with team "blue", with behavior CoachBehavior()

opponent = new Player at (0, Uniform(4, 6), 0), with name "defender1",
            with behavior DefenderBehavior()

goal = new Goal at (0, 17, 0)

'''

In [57]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_overlap_synth_add_node, response_overlap_synth_add_node = fixer.run()

In [58]:
response_overlap_synth_add_node

'### Brief Explanation of Changes\n\n*   **What the feedback was:**\n    The coach pointed out that the original program incorrectly assumes the teammate will always pass the ball after the coach performs an overlapping run. The feedback stated that if the teammate decides to dribble towards the goal instead, the coach should reposition to "create an angle of pass near the goal" to provide support.\n\n*   **What was changed in the Scenic code:**\n    To address the feedback, I modified the `CoachBehavior` to handle this branching logic:\n    1.  **New Termination Condition:** I added a second termination condition to the initial `do MoveTo(...)` action. The action now stops not only if the teammate passes (`MakePass`), but also if the teammate starts dribbling towards the goal (`MovingTowards`). This is implemented with a new constraint object `A1termination_1` and its lambda function `λ_termination1`.\n    2.  **Conditional Logic:** I introduced an `if/else` statement after the initia

In [59]:
fixed_code_overlap_synth_add_node

'python\nbehavior CoachBehavior():\n    do Speak("I will make an overlapping run to create space and attract the defender.")\n    # MODIFIED: The run now terminates if the teammate passes OR if they start dribbling to the goal.\n    # This addresses the feedback that the teammate might not pass and may dribble to the goal instead.\n    do MoveTo(λ_target0()) until λ_termination0(simulation(), None) or λ_termination1(simulation(), None)\n\n    # ADDED: A conditional branch to handle the two possible outcomes from the previous state.\n    if λ_termination0(simulation(), None):\n        # This is the original path: the teammate passed the ball.\n        do Speak("The teammate is passing. I will receive the ball.")\n        do ReceiveBall()\n        if λ_precondition1(simulation(), None):\n            do Speak("I have enough space, so I will dribble towards the goal.")\n            do MoveTo(goal)\n        else:\n            do Speak("The defender is too close, so I will pass the ball back

## Delete node

In [61]:
ORIGIN_NAME = 'overlap-Nishant'
SYNTH_NAME = 'overlap-Nishant-synth-feedback-delete-node'

In [62]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/6 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 1299
Duration of Video: 43.3 seconds


Importing:  17%|█████▋                            | 1/6 [00:02<00:13,  2.66s/it]

Total # of frames saved: 44
FPS: 30.0
Samplerate: 1
Total # of frames: 962
Duration of Video: 32.06666666666667 seconds


Importing:  33%|███████████▎                      | 2/6 [00:04<00:09,  2.28s/it]

...

Total # of frames saved: 21


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 1641
Duration of Video: 27.35 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:02<00:00,  2.12s/it]

Total # of frames saved: 28


In [63]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After receiving the ball we want to always pass back to the teammate. Don’t go for a goal even if opponent is far from you."
code = '''behavior CoachBehavior():
    do Speak("I will make an overlapping run to create space and attract the defender.")
    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)
    do Speak("The teammate is passing. I will receive the ball.")
    do ReceiveBall()
    if λ_precondition1(simulation(), None):
        do Speak("I have enough space, so I will dribble towards the goal.")
        do MoveTo(goal)
    else:
        do Speak("The defender is too close, so I will pass the ball back to my teammate.")
        do Pass(teammate)

A1target_0 = Overlap({'player': 'Coach', 'ball': 'ball', 'goal': 'goal', 'opponent': 'defender1', 'theta': {'avg': 37.49500473945939, 'std': 4.316867385966453}, 'dist': {'avg': 4.144414138072041, 'std': 1.1398863177626926}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_1 = DistanceTo({'to': 'defender1', 'from': 'Coach', 'operator': 'greater_than', 'min': {'avg': 4.238624128913989, 'std': 0.613329061405903}})
A2precondition_1 = HasBallPossession({'player': 'Coach'})

def λ_target0():
    return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
    return A1termination_0.bool(simulation())

def λ_precondition1(scene, sample):
    return A1precondition_1.bool(simulation()) and A2precondition_1.bool(simulation())
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I will make an overlapping run to create space and attract the defender.")
    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)
    do Speak("The teammate is passing. I will receive the ball.")
    do ReceiveBall()
    if λ_precondition1(simulation(), None):
        do Speak("I have enough space, so I will dribble towards the goal.")
        do MoveTo(goal)
    else:
        do Speak("The defender is too close, so I will pass the ball back to my teammate.")
        do Pass(teammate)

A1target_0 = Overlap({'player': 'Coach', 'ball': 'ball', 'goal': 'goal', 'opponent': 'defender1', 'theta': {'avg': 37.49500473945939, 'std': 4.316867385966453}, 'dist': {'avg': 4.144414138072041, 'std': 1.1398863177626926}})
A1termination_0 = MakePass({'player': 'teammate'})
A1precondition_1 = DistanceTo({'to': 'defender1', 'from': 'Coach', 'operator': 'greater_than', 'min': {'avg': 4.238624128913989, 'std': 0.613329061405903}})
A2precondition_1 = HasBallPossession({'player': 'Coach'})

def λ_target0():
    return A1target_0.dist(simulation(), ego=True)

def λ_termination0(scene, sample):
    return A1termination_0.bool(simulation())

def λ_precondition1(scene, sample):
    return A1precondition_1.bool(simulation()) and A2precondition_1.bool(simulation())




opponent_y_distance = Uniform(3, 5)
opponent_x_distance = Uniform(-2, 2)
ego_x_distance = Uniform(-2, 2)
ego_y_distance = Uniform(-1, -2)

# Ensure teammate and opponent are on the same side
#require (opponent_x_distance < 0 and ego_x_distance < 0) or (opponent_x_distance >= 0 and ego_x_distance >= 0)

behavior Follow(obj):
    while ego.position.y > 1:
        do MoveToBehavior(obj, distance = 2, status = f"Follow {obj.name}")

behavior TeammateBehavior():
    do Idle() for 1 seconds
    do GetBallPossession(ball)
    do Idle() until ego.position.y > 2
    print("ego at good position")
    do Idle() for 1 seconds
    do Pass(ego, slow=False) until (distance from opponent to ego) <= 3
    print("pass happened")
    do DribbleTo(goal) until (distance from opponent to ego) > 3
    do Idle() for 2 seconds
    

behavior DefenderBehavior():
    do Idle() for 1 seconds
    do Idle() until ego.position.y > 1
    do Follow(ego) until ego.gameObject.ballPossession
    

teammate = new Player at (0, 0, 0),
      with behavior TeammateBehavior(), with name "teammate"

ball = new Ball ahead of teammate by 1

ego = new Coach at (ego_x_distance, ego_y_distance, 0), with name "Coach", with team "blue", with behavior CoachBehavior()

opponent = new Player at (0, Uniform(4, 6), 0), with name "defender1",
            with behavior DefenderBehavior()

goal = new Goal at (0, 17, 0)

'''

In [64]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_overlap_delete_node, response_overlap_delete_node = fixer.run()

In [65]:
response_overlap_delete_node

'### Feedback Analysis and Code Changes\n\n**1. Coach\'s Feedback:**\nThe coach\'s feedback was that after the coach avatar receives the ball, it should *always* pass the ball back to the teammate. The current program has a conditional behavior where the coach dribbles towards the goal if there is enough space (i.e., the defender is far away). The coach wants to eliminate this conditional dribbling and enforce a pass-back action in all cases.\n\n**2. Implemented Changes:**\nTo address the feedback, I made the following changes to the `CoachBehavior()` and its supporting definitions:\n- In the `CoachBehavior` function, I removed the `if/else` block that checked the distance to the defender (`λ_precondition1`).\n- I replaced this conditional logic with an unconditional `Pass(teammate)` action, which is always executed after `ReceiveBall()`.\n- I updated the `Speak` action to reflect this new, non-conditional behavior.\n- Consequently, the `λ_precondition1` function and its associated con

In [66]:
fixed_code_overlap_delete_node

'python\nbehavior CoachBehavior():\n    do Speak("I will make an overlapping run to create space and attract the defender.")\n    do MoveTo(λ_target0()) until λ_termination0(simulation(), None)\n    do Speak("The teammate is passing. I will receive the ball.")\n    do ReceiveBall()\n    # CHANGE: Removed the conditional logic that decided between dribbling to the goal or passing.\n    # Based on the feedback, the coach now always passes back to the teammate after receiving the ball.\n    do Speak("I will pass the ball back to my teammate.")\n    do Pass(teammate)\n\nA1target_0 = Overlap({\'player\': \'Coach\', \'ball\': \'ball\', \'goal\': \'goal\', \'opponent\': \'defender1\', \'theta\': {\'avg\': 37.49500473945939, \'std\': 4.316867385966453}, \'dist\': {\'avg\': 4.144414138072041, \'std\': 1.1398863177626926}})\nA1termination_0 = MakePass({\'player\': \'teammate\'})\n# CHANGE: Removed A1precondition_1 and A2precondition_1 as they were used by the\n# conditional logic that has been r

# Distribute Nishant

## Add edge

In [84]:
ORIGIN_NAME = 'distribute-Nishant'
SYNTH_NAME = 'distribute-Nishant-synth-feedback-add-edge'

In [86]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/5 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 795
Duration of Video: 26.5 seconds


Importing:  20%|██████▊                           | 1/5 [00:01<00:07,  1.78s/it]

Total # of frames saved: 27
FPS: 30.0
Samplerate: 1
Total # of frames: 909
Duration of Video: 30.3 seconds


Importing:  40%|█████████████▌                    | 2/5 [00:03<00:05,  1.93s/it]

...

Total # of frames saved: 29


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 1986
Duration of Video: 33.1 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:02<00:00,  2.73s/it]

Total # of frames saved: 34


In [87]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After you move to the left Winger, and you don't receive the ball, move to the left striker."
code = '''behavior CoachBehavior():
    do Speak("I'm waiting for the pass, ready to receive the ball.")
    #do ReceiveBall()
    do GetBallPossession(ball)
    do Speak("I have possession. Now I'm looking for the best passing option.")
    if λ_precondition_pass_to_RS(simulation(), None):
        do Speak("Right Striker is open. I'm passing the ball to them now.")
        do Pass(RightStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving to support the Right Striker.")
        do MoveTo(λ_target_support_RS())
    elif λ_precondition_pass_to_LS(simulation(), None):
        do Speak("Left Striker has space. I'm sending the ball to them.")
        do Pass(LeftStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. Moving up to support the Left Striker.")
        do MoveTo(λ_target_support_LS())
    elif λ_precondition_pass_to_LW(simulation(), None):
        do Speak("Left Winger is the best option. Passing it wide.")
        do Pass(LeftWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. Now I'll provide support for the Left Winger.")
        do MoveTo(λ_target_support_LW())
    elif λ_precondition_pass_to_RW(simulation(), None):
        do Speak("Right Winger is available. Passing the ball to them.")
        do Pass(RightWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving forward to support the Right Winger.")
        do MoveTo(λ_target_support_RW())
    else:
        do Speak("No good options. I'll hold the ball and wait for an opening.")
        do Idle()
    do Speak("I am in a good supporting position now.")
    do Idle()

A1precondition_pass_to_RS = HasPath({'obj1': 'Coach', 'obj2': 'RightStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LS = HasPath({'obj1': 'Coach', 'obj2': 'LeftStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LW = HasPath({'obj1': 'Coach', 'obj2': 'LeftWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_RW = HasPath({'obj1': 'Coach', 'obj2': 'RightWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1target_support_RS = CloseTo({'obj': 'Coach', 'ref': 'RightStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LS = CloseTo({'obj': 'Coach', 'ref': 'LeftStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LW = CloseTo({'obj': 'Coach', 'ref': 'LeftWinger', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_RW = CloseTo({'obj': 'Coach', 'ref': 'RightWinger', 'max': {'avg': 6.0, 'std': 1.0}})

def λ_precondition_pass_to_RS(scene, sample):
    return A1precondition_pass_to_RS.bool(simulation())

def λ_precondition_pass_to_LS(scene, sample):
    return A1precondition_pass_to_LS.bool(simulation())

def λ_precondition_pass_to_LW(scene, sample):
    return A1precondition_pass_to_LW.bool(simulation())

def λ_precondition_pass_to_RW(scene, sample):
    return A1precondition_pass_to_RW.bool(simulation())

def λ_target_support_RS():
    return A1target_support_RS.dist(simulation(), ego=True)

def λ_target_support_LS():
    return A1target_support_LS.dist(simulation(), ego=True)

def λ_target_support_LW():
    return A1target_support_LW.dist(simulation(), ego=True)

def λ_target_support_RW():
    return A1target_support_RW.dist(simulation(), ego=True)

def λ_termination(scene, sample):
    return False
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I'm waiting for the pass, ready to receive the ball.")
    #do ReceiveBall()
    do GetBallPossession(ball)
    do Speak("I have possession. Now I'm looking for the best passing option.")
    if λ_precondition_pass_to_RS(simulation(), None):
        do Speak("Right Striker is open. I'm passing the ball to them now.")
        do Pass(RightStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving to support the Right Striker.")
        do MoveTo(λ_target_support_RS())
    elif λ_precondition_pass_to_LS(simulation(), None):
        do Speak("Left Striker has space. I'm sending the ball to them.")
        do Pass(LeftStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. Moving up to support the Left Striker.")
        do MoveTo(λ_target_support_LS())
    elif λ_precondition_pass_to_LW(simulation(), None):
        do Speak("Left Winger is the best option. Passing it wide.")
        do Pass(LeftWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. Now I'll provide support for the Left Winger.")
        do MoveTo(λ_target_support_LW())
    elif λ_precondition_pass_to_RW(simulation(), None):
        do Speak("Right Winger is available. Passing the ball to them.")
        do Pass(RightWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving forward to support the Right Winger.")
        do MoveTo(λ_target_support_RW())
    else:
        do Speak("No good options. I'll hold the ball and wait for an opening.")
        do Idle()
    do Speak("I am in a good supporting position now.")
    do Idle()

A1precondition_pass_to_RS = HasPath({'obj1': 'Coach', 'obj2': 'RightStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LS = HasPath({'obj1': 'Coach', 'obj2': 'LeftStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LW = HasPath({'obj1': 'Coach', 'obj2': 'LeftWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_RW = HasPath({'obj1': 'Coach', 'obj2': 'RightWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1target_support_RS = CloseTo({'obj': 'Coach', 'ref': 'RightStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LS = CloseTo({'obj': 'Coach', 'ref': 'LeftStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LW = CloseTo({'obj': 'Coach', 'ref': 'LeftWinger', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_RW = CloseTo({'obj': 'Coach', 'ref': 'RightWinger', 'max': {'avg': 6.0, 'std': 1.0}})

def λ_precondition_pass_to_RS(scene, sample):
    return A1precondition_pass_to_RS.bool(simulation())

def λ_precondition_pass_to_LS(scene, sample):
    return A1precondition_pass_to_LS.bool(simulation())

def λ_precondition_pass_to_LW(scene, sample):
    return A1precondition_pass_to_LW.bool(simulation())

def λ_precondition_pass_to_RW(scene, sample):
    return A1precondition_pass_to_RW.bool(simulation())

def λ_target_support_RS():
    return A1target_support_RS.dist(simulation(), ego=True)

def λ_target_support_LS():
    return A1target_support_LS.dist(simulation(), ego=True)

def λ_target_support_LW():
    return A1target_support_LW.dist(simulation(), ego=True)

def λ_target_support_RW():
    return A1target_support_RW.dist(simulation(), ego=True)

def λ_termination(scene, sample):
    return False



# Ego (center midfielder) at origin
pi = 3.1415
ego = new Coach at (0, 0, 0), facing toward (0, 0, 0), with team "blue", with behavior CoachBehavior()

# Wingers
left_winger_angle = 90 + Uniform(0, 10)  # degrees from y-axis, 90 is positive x-axis (left), variance +/-10
right_winger_angle = -90 + Uniform(0, 10)  # degrees from y-axis, -90 is negative x-axis (right), variance +/-10
winger_dist = Uniform(6,8)

left_winger_x = winger_dist * sin(left_winger_angle * pi / 180)
left_winger_y = winger_dist * cos(left_winger_angle * pi / 180)
LeftWinger = new Player at (left_winger_x, left_winger_y, 0), facing toward ego, with name "LeftWinger", with team "blue"

right_winger_x = winger_dist * sin(right_winger_angle * pi / 180)
right_winger_y = winger_dist * cos(right_winger_angle * pi / 180)
RightWinger = new Player at (right_winger_x, right_winger_y, 0), facing toward ego, with name "RightWinger", with team "blue"

# Strikers
left_striker_angle = -Uniform(8, 20)
right_striker_angle = Uniform(8, 20)
striker_dist = Uniform(8,10)

left_striker_x = striker_dist * sin(left_striker_angle * pi / 180)
left_striker_y = striker_dist * cos(left_striker_angle * pi / 180)
LeftStriker = new Player at (left_striker_x, left_striker_y, 0), facing toward ego, with name "LeftStriker", with team "blue"

right_striker_x = striker_dist * sin(right_striker_angle * pi / 180)
right_striker_y = striker_dist * cos(right_striker_angle * pi / 180)
RightStriker = new Player at (right_striker_x, right_striker_y, 0), facing toward ego, with name "RightStriker", with team "blue"

# Ball at ego's feet
ball = new Ball at (0, 1, 0)

# Defenders: each assigned to one attacker, at a distance and angle in front of them, facing ego
# Helper function for defender placement
# (Scenic doesn't support functions in .scenic, so we inline the logic)

defender1_angle = Uniform(-10, 10)
defender1_dist = Uniform(2,4)
defender1_x = ego.position.x + defender1_dist * sin(defender1_angle * pi / 180)
defender1_y = ego.position.y + defender1_dist * cos(defender1_angle * pi / 180)
defender1 = new Player at (defender1_x, defender1_y, 0), facing toward ego, with team "red", with name "Defender1"

defender2_angle = Uniform(-30, 30)
defender2_dist = Uniform(1,2)
defender2_x = LeftWinger.position.x + defender2_dist * sin(defender2_angle * pi / 180)
defender2_y = LeftWinger.position.y + defender2_dist * cos(defender2_angle * pi / 180)
defender2 = new Player at (defender2_x, defender2_y, 0), facing toward ego, with team "red", with name "Defender2"

defender3_angle = Uniform(-30, 30)
defender3_dist = Uniform(1,2)
defender3_x = RightWinger.position.x + defender3_dist * sin(defender3_angle * pi / 180)
defender3_y = RightWinger.position.y + defender3_dist * cos(defender3_angle * pi / 180)
defender3 = new Player at (defender3_x, defender3_y, 0), facing toward ego, with team "red", with name "Defender3"

defender4_angle = Uniform(-30, 30)
defender4_dist = Uniform(1,2)
defender4_x = LeftStriker.position.x + defender4_dist * sin(defender4_angle * pi / 180)
defender4_y = LeftStriker.position.y + defender4_dist * cos(defender4_angle * pi / 180)
defender4 = new Player at (defender4_x, defender4_y, 0), facing toward ego, with team "red", with name "Defender4"

defender5_angle = Uniform(-30, 30)
defender5_dist = Uniform(1,2)
defender5_x = RightStriker.position.x + defender5_dist * sin(defender5_angle * pi / 180)
defender5_y = RightStriker.position.y + defender5_dist * cos(defender5_angle * pi / 180)
defender5 = new Player at (defender5_x, defender5_y, 0), facing toward ego, with team "red", with name "Defender5"

'''

In [88]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_distribute_add_edge, response_distribute_add_edge = fixer.run()

In [89]:
response_distribute_add_edge

'Based on the coach\'s feedback, I will adjust the `CoachBehavior` to add a new action sequence.\n\n- **The Feedback:** The coach specified that after the agent passes the ball to the Left Winger and moves to a supporting position, it shouldn\'t just become idle. If a return pass isn\'t made, the agent should reposition itself to support the Left Striker.\n\n- **My Changes:** I have modified the specific `elif` block in the `CoachBehavior` function that handles the scenario of passing to the Left Winger. After the initial `MoveTo` action to support the Left Winger, I\'ve added a new `MoveTo` action to reposition the coach to support the Left Striker, as requested. I also included a `Speak` action to narrate this new behavior, making the agent\'s intention clear.\n\nHere is the corrected code snippet:\n\n```python\nbehavior CoachBehavior():\n    do Speak("I\'m waiting for the pass, ready to receive the ball.")\n    #do ReceiveBall()\n    do GetBallPossession(ball)\n    do Speak("I have 

In [90]:
fixed_code_distribute_add_edge

'python\nbehavior CoachBehavior():\n    do Speak("I\'m waiting for the pass, ready to receive the ball.")\n    #do ReceiveBall()\n    do GetBallPossession(ball)\n    do Speak("I have possession. Now I\'m looking for the best passing option.")\n    if λ_precondition_pass_to_RS(simulation(), None):\n        do Speak("Right Striker is open. I\'m passing the ball to them now.")\n        do Pass(RightStriker)\n        do Idle() for 2 seconds\n        do Speak("Pass complete. I\'m moving to support the Right Striker.")\n        do MoveTo(λ_target_support_RS())\n    elif λ_precondition_pass_to_LS(simulation(), None):\n        do Speak("Left Striker has space. I\'m sending the ball to them.")\n        do Pass(LeftStriker)\n        do Idle() for 2 seconds\n        do Speak("Pass complete. Moving up to support the Left Striker.")\n        do MoveTo(λ_target_support_LS())\n    elif λ_precondition_pass_to_LW(simulation(), None):\n        do Speak("Left Winger is the best option. Passing it wide.")

## Add node

In [92]:
ORIGIN_NAME = 'distribute-Nishant'
SYNTH_NAME = 'distribute-Nishant-synth-feedback-add-node'

In [93]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/5 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 795
Duration of Video: 26.5 seconds


Importing:  20%|██████▊                           | 1/5 [00:01<00:06,  1.74s/it]

Total # of frames saved: 27
FPS: 30.0
Samplerate: 1
Total # of frames: 909
Duration of Video: 30.3 seconds


Importing:  40%|█████████████▌                    | 2/5 [00:03<00:05,  1.90s/it]

...

Total # of frames saved: 29


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 1908
Duration of Video: 31.8 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:02<00:00,  2.53s/it]

Total # of frames saved: 32


In [94]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After you move to the right striker, left striker, right winger, or left winger after you pass to them, if the defender is close to your teammate that has the ball, your teammate will pass it to you and you should receive the ball from them."
code = '''behavior CoachBehavior():
    do Speak("I'm waiting for the pass, ready to receive the ball.")
    #do ReceiveBall()
    do GetBallPossession(ball)
    do Speak("I have possession. Now I'm looking for the best passing option.")
    if λ_precondition_pass_to_RS(simulation(), None):
        do Speak("Right Striker is open. I'm passing the ball to them now.")
        do Pass(RightStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving to support the Right Striker.")
        do MoveTo(λ_target_support_RS())
    elif λ_precondition_pass_to_LS(simulation(), None):
        do Speak("Left Striker has space. I'm sending the ball to them.")
        do Pass(LeftStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. Moving up to support the Left Striker.")
        do MoveTo(λ_target_support_LS())
    elif λ_precondition_pass_to_LW(simulation(), None):
        do Speak("Left Winger is the best option. Passing it wide.")
        do Pass(LeftWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. Now I'll provide support for the Left Winger.")
        do MoveTo(λ_target_support_LW())
    elif λ_precondition_pass_to_RW(simulation(), None):
        do Speak("Right Winger is available. Passing the ball to them.")
        do Pass(RightWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving forward to support the Right Winger.")
        do MoveTo(λ_target_support_RW())
    else:
        do Speak("No good options. I'll hold the ball and wait for an opening.")
        do Idle()
    do Speak("I am in a good supporting position now.")
    do Idle()

A1precondition_pass_to_RS = HasPath({'obj1': 'Coach', 'obj2': 'RightStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LS = HasPath({'obj1': 'Coach', 'obj2': 'LeftStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LW = HasPath({'obj1': 'Coach', 'obj2': 'LeftWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_RW = HasPath({'obj1': 'Coach', 'obj2': 'RightWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1target_support_RS = CloseTo({'obj': 'Coach', 'ref': 'RightStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LS = CloseTo({'obj': 'Coach', 'ref': 'LeftStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LW = CloseTo({'obj': 'Coach', 'ref': 'LeftWinger', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_RW = CloseTo({'obj': 'Coach', 'ref': 'RightWinger', 'max': {'avg': 6.0, 'std': 1.0}})

def λ_precondition_pass_to_RS(scene, sample):
    return A1precondition_pass_to_RS.bool(simulation())

def λ_precondition_pass_to_LS(scene, sample):
    return A1precondition_pass_to_LS.bool(simulation())

def λ_precondition_pass_to_LW(scene, sample):
    return A1precondition_pass_to_LW.bool(simulation())

def λ_precondition_pass_to_RW(scene, sample):
    return A1precondition_pass_to_RW.bool(simulation())

def λ_target_support_RS():
    return A1target_support_RS.dist(simulation(), ego=True)

def λ_target_support_LS():
    return A1target_support_LS.dist(simulation(), ego=True)

def λ_target_support_LW():
    return A1target_support_LW.dist(simulation(), ego=True)

def λ_target_support_RW():
    return A1target_support_RW.dist(simulation(), ego=True)

def λ_termination(scene, sample):
    return False
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I'm waiting for the pass, ready to receive the ball.")
    #do ReceiveBall()
    do GetBallPossession(ball)
    do Speak("I have possession. Now I'm looking for the best passing option.")
    if λ_precondition_pass_to_RS(simulation(), None):
        do Speak("Right Striker is open. I'm passing the ball to them now.")
        do Pass(RightStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving to support the Right Striker.")
        do MoveTo(λ_target_support_RS())
    elif λ_precondition_pass_to_LS(simulation(), None):
        do Speak("Left Striker has space. I'm sending the ball to them.")
        do Pass(LeftStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. Moving up to support the Left Striker.")
        do MoveTo(λ_target_support_LS())
    elif λ_precondition_pass_to_LW(simulation(), None):
        do Speak("Left Winger is the best option. Passing it wide.")
        do Pass(LeftWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. Now I'll provide support for the Left Winger.")
        do MoveTo(λ_target_support_LW())
    elif λ_precondition_pass_to_RW(simulation(), None):
        do Speak("Right Winger is available. Passing the ball to them.")
        do Pass(RightWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving forward to support the Right Winger.")
        do MoveTo(λ_target_support_RW())
    else:
        do Speak("No good options. I'll hold the ball and wait for an opening.")
        do Idle()
    do Speak("I am in a good supporting position now.")
    do Idle()

A1precondition_pass_to_RS = HasPath({'obj1': 'Coach', 'obj2': 'RightStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LS = HasPath({'obj1': 'Coach', 'obj2': 'LeftStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LW = HasPath({'obj1': 'Coach', 'obj2': 'LeftWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_RW = HasPath({'obj1': 'Coach', 'obj2': 'RightWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1target_support_RS = CloseTo({'obj': 'Coach', 'ref': 'RightStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LS = CloseTo({'obj': 'Coach', 'ref': 'LeftStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LW = CloseTo({'obj': 'Coach', 'ref': 'LeftWinger', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_RW = CloseTo({'obj': 'Coach', 'ref': 'RightWinger', 'max': {'avg': 6.0, 'std': 1.0}})

def λ_precondition_pass_to_RS(scene, sample):
    return A1precondition_pass_to_RS.bool(simulation())

def λ_precondition_pass_to_LS(scene, sample):
    return A1precondition_pass_to_LS.bool(simulation())

def λ_precondition_pass_to_LW(scene, sample):
    return A1precondition_pass_to_LW.bool(simulation())

def λ_precondition_pass_to_RW(scene, sample):
    return A1precondition_pass_to_RW.bool(simulation())

def λ_target_support_RS():
    return A1target_support_RS.dist(simulation(), ego=True)

def λ_target_support_LS():
    return A1target_support_LS.dist(simulation(), ego=True)

def λ_target_support_LW():
    return A1target_support_LW.dist(simulation(), ego=True)

def λ_target_support_RW():
    return A1target_support_RW.dist(simulation(), ego=True)

def λ_termination(scene, sample):
    return False



# Ego (center midfielder) at origin
pi = 3.1415
ego = new Coach at (0, 0, 0), facing toward (0, 0, 0), with team "blue", with behavior CoachBehavior()

# Wingers
left_winger_angle = 90 + Uniform(0, 10)  # degrees from y-axis, 90 is positive x-axis (left), variance +/-10
right_winger_angle = -90 + Uniform(0, 10)  # degrees from y-axis, -90 is negative x-axis (right), variance +/-10
winger_dist = Uniform(6,8)

left_winger_x = winger_dist * sin(left_winger_angle * pi / 180)
left_winger_y = winger_dist * cos(left_winger_angle * pi / 180)
LeftWinger = new Player at (left_winger_x, left_winger_y, 0), facing toward ego, with name "LeftWinger", with team "blue"

right_winger_x = winger_dist * sin(right_winger_angle * pi / 180)
right_winger_y = winger_dist * cos(right_winger_angle * pi / 180)
RightWinger = new Player at (right_winger_x, right_winger_y, 0), facing toward ego, with name "RightWinger", with team "blue"

# Strikers
left_striker_angle = -Uniform(8, 20)
right_striker_angle = Uniform(8, 20)
striker_dist = Uniform(8,10)

left_striker_x = striker_dist * sin(left_striker_angle * pi / 180)
left_striker_y = striker_dist * cos(left_striker_angle * pi / 180)
LeftStriker = new Player at (left_striker_x, left_striker_y, 0), facing toward ego, with name "LeftStriker", with team "blue"

right_striker_x = striker_dist * sin(right_striker_angle * pi / 180)
right_striker_y = striker_dist * cos(right_striker_angle * pi / 180)
RightStriker = new Player at (right_striker_x, right_striker_y, 0), facing toward ego, with name "RightStriker", with team "blue"

# Ball at ego's feet
ball = new Ball at (0, 1, 0)

# Defenders: each assigned to one attacker, at a distance and angle in front of them, facing ego
# Helper function for defender placement
# (Scenic doesn't support functions in .scenic, so we inline the logic)

defender1_angle = Uniform(-10, 10)
defender1_dist = Uniform(2,4)
defender1_x = ego.position.x + defender1_dist * sin(defender1_angle * pi / 180)
defender1_y = ego.position.y + defender1_dist * cos(defender1_angle * pi / 180)
defender1 = new Player at (defender1_x, defender1_y, 0), facing toward ego, with team "red", with name "Defender1"

defender2_angle = Uniform(-30, 30)
defender2_dist = Uniform(1,2)
defender2_x = LeftWinger.position.x + defender2_dist * sin(defender2_angle * pi / 180)
defender2_y = LeftWinger.position.y + defender2_dist * cos(defender2_angle * pi / 180)
defender2 = new Player at (defender2_x, defender2_y, 0), facing toward ego, with team "red", with name "Defender2"

defender3_angle = Uniform(-30, 30)
defender3_dist = Uniform(1,2)
defender3_x = RightWinger.position.x + defender3_dist * sin(defender3_angle * pi / 180)
defender3_y = RightWinger.position.y + defender3_dist * cos(defender3_angle * pi / 180)
defender3 = new Player at (defender3_x, defender3_y, 0), facing toward ego, with team "red", with name "Defender3"

defender4_angle = Uniform(-30, 30)
defender4_dist = Uniform(1,2)
defender4_x = LeftStriker.position.x + defender4_dist * sin(defender4_angle * pi / 180)
defender4_y = LeftStriker.position.y + defender4_dist * cos(defender4_angle * pi / 180)
defender4 = new Player at (defender4_x, defender4_y, 0), facing toward ego, with team "red", with name "Defender4"

defender5_angle = Uniform(-30, 30)
defender5_dist = Uniform(1,2)
defender5_x = RightStriker.position.x + defender5_dist * sin(defender5_angle * pi / 180)
defender5_y = RightStriker.position.y + defender5_dist * cos(defender5_angle * pi / 180)
defender5 = new Player at (defender5_x, defender5_y, 0), facing toward ego, with team "red", with name "Defender5"

'''

In [95]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_distribute_add_node, response_distribute_add_node = fixer.run()

In [96]:
response_distribute_add_node

'### Brief Explanation of Changes\n\nThe coach\'s feedback indicated that after the coach avatar passes the ball to a teammate (Right/Left Striker or Right/Left Winger) and moves to a supporting position, it should be prepared to receive a pass back if the teammate is under pressure from a defender.\n\nTo implement this, I made the following changes:\n\n1.  **Added New Constraints:** I introduced four new `CloseTo` constraints (`A2defender_near_RS`, `A2defender_near_LS`, `A2defender_near_LW`, `A2defender_near_RW`). Each constraint checks if a specific defender is close to the corresponding teammate who has the ball. The defender-teammate pairings were determined from the object placement logic in the full Scenic code.\n\n2.  **Added New Lambda Functions:** I created corresponding lambda functions (`λ_defender_near_RS`, `λ_defender_near_LS`, etc.) to evaluate these new `CloseTo` constraints during the simulation.\n\n3.  **Modified `CoachBehavior`:** Inside each of the `if/elif` blocks t

In [97]:
fixed_code_distribute_add_node

'python\nbehavior CoachBehavior():\n    do Speak("I\'m waiting for the pass, ready to receive the ball.")\n    #do ReceiveBall()\n    do GetBallPossession(ball)\n    do Speak("I have possession. Now I\'m looking for the best passing option.")\n    if λ_precondition_pass_to_RS(simulation(), None):\n        do Speak("Right Striker is open. I\'m passing the ball to them now.")\n        do Pass(RightStriker)\n        do Idle() for 2 seconds\n        do Speak("Pass complete. I\'m moving to support the Right Striker.")\n        do MoveTo(λ_target_support_RS())\n        # CHANGE: Added a check to see if the teammate is under pressure from a defender.\n        # If so, the coach should receive a pass back from the teammate.\n        if λ_defender_near_RS(simulation(), None):\n            do Speak("Right Striker is under pressure, passing back to me.")\n            do GetBallPossession(ball)\n    elif λ_precondition_pass_to_LS(simulation(), None):\n        do Speak("Left Striker has space. I\'m

## Delete node

In [99]:
ORIGIN_NAME = 'distribute-Nishant'
SYNTH_NAME = 'distribute-Nishant-synth-feedback-delete-node'

In [100]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{ORIGIN_NAME}'
EXAMPLE_DIR = dir + f'/data/{SYNTH_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)
synth_demo = UnityTranslator.get_from(EXAMPLE_DIR, sample_rate=1)

Importing:   0%|                                          | 0/5 [00:00<?, ?it/s]

FPS: 30.0
Samplerate: 1
Total # of frames: 795
Duration of Video: 26.5 seconds


Importing:  20%|██████▊                           | 1/5 [00:01<00:06,  1.74s/it]

Total # of frames saved: 27
FPS: 30.0
Samplerate: 1
Total # of frames: 909
Duration of Video: 30.3 seconds


Importing:  40%|█████████████▌                    | 2/5 [00:03<00:05,  1.91s/it]

...

Total # of frames saved: 29


Importing:   0%|                                          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 1
Total # of frames: 892
Duration of Video: 14.866666666666667 seconds


Importing: 100%|██████████████████████████████████| 1/1 [00:01<00:00,  1.22s/it]

Total # of frames saved: 15


In [101]:
url = 'https://docs.scenic-lang.org/en/latest/syntax_guide.html'
feedback = "After you pass it to your right striker or left striker, do not move up the field to support them as the defender in front of you can easily follow you."
code = '''behavior CoachBehavior():
    do Speak("I'm waiting for the pass, ready to receive the ball.")
    #do ReceiveBall()
    do GetBallPossession(ball)
    do Speak("I have possession. Now I'm looking for the best passing option.")
    if λ_precondition_pass_to_RS(simulation(), None):
        do Speak("Right Striker is open. I'm passing the ball to them now.")
        do Pass(RightStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving to support the Right Striker.")
        do MoveTo(λ_target_support_RS())
    elif λ_precondition_pass_to_LS(simulation(), None):
        do Speak("Left Striker has space. I'm sending the ball to them.")
        do Pass(LeftStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. Moving up to support the Left Striker.")
        do MoveTo(λ_target_support_LS())
    elif λ_precondition_pass_to_LW(simulation(), None):
        do Speak("Left Winger is the best option. Passing it wide.")
        do Pass(LeftWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. Now I'll provide support for the Left Winger.")
        do MoveTo(λ_target_support_LW())
    elif λ_precondition_pass_to_RW(simulation(), None):
        do Speak("Right Winger is available. Passing the ball to them.")
        do Pass(RightWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving forward to support the Right Winger.")
        do MoveTo(λ_target_support_RW())
    else:
        do Speak("No good options. I'll hold the ball and wait for an opening.")
        do Idle()
    do Speak("I am in a good supporting position now.")
    do Idle()

A1precondition_pass_to_RS = HasPath({'obj1': 'Coach', 'obj2': 'RightStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LS = HasPath({'obj1': 'Coach', 'obj2': 'LeftStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LW = HasPath({'obj1': 'Coach', 'obj2': 'LeftWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_RW = HasPath({'obj1': 'Coach', 'obj2': 'RightWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1target_support_RS = CloseTo({'obj': 'Coach', 'ref': 'RightStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LS = CloseTo({'obj': 'Coach', 'ref': 'LeftStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LW = CloseTo({'obj': 'Coach', 'ref': 'LeftWinger', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_RW = CloseTo({'obj': 'Coach', 'ref': 'RightWinger', 'max': {'avg': 6.0, 'std': 1.0}})

def λ_precondition_pass_to_RS(scene, sample):
    return A1precondition_pass_to_RS.bool(simulation())

def λ_precondition_pass_to_LS(scene, sample):
    return A1precondition_pass_to_LS.bool(simulation())

def λ_precondition_pass_to_LW(scene, sample):
    return A1precondition_pass_to_LW.bool(simulation())

def λ_precondition_pass_to_RW(scene, sample):
    return A1precondition_pass_to_RW.bool(simulation())

def λ_target_support_RS():
    return A1target_support_RS.dist(simulation(), ego=True)

def λ_target_support_LS():
    return A1target_support_LS.dist(simulation(), ego=True)

def λ_target_support_LW():
    return A1target_support_LW.dist(simulation(), ego=True)

def λ_target_support_RW():
    return A1target_support_RW.dist(simulation(), ego=True)

def λ_termination(scene, sample):
    return False
'''

context = '''
from scenic.simulators.unity.actions import *
from scenic.simulators.unity.behaviors import *
from scenic.simulators.unity.constraints import *
model scenic.simulators.unity.model
import trimesh
from scenic.core.regions import MeshVolumeRegion
import random

behavior CoachBehavior():
    do Speak("I'm waiting for the pass, ready to receive the ball.")
    #do ReceiveBall()
    do GetBallPossession(ball)
    do Speak("I have possession. Now I'm looking for the best passing option.")
    if λ_precondition_pass_to_RS(simulation(), None):
        do Speak("Right Striker is open. I'm passing the ball to them now.")
        do Pass(RightStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving to support the Right Striker.")
        do MoveTo(λ_target_support_RS())
    elif λ_precondition_pass_to_LS(simulation(), None):
        do Speak("Left Striker has space. I'm sending the ball to them.")
        do Pass(LeftStriker)
        do Idle() for 2 seconds
        do Speak("Pass complete. Moving up to support the Left Striker.")
        do MoveTo(λ_target_support_LS())
    elif λ_precondition_pass_to_LW(simulation(), None):
        do Speak("Left Winger is the best option. Passing it wide.")
        do Pass(LeftWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. Now I'll provide support for the Left Winger.")
        do MoveTo(λ_target_support_LW())
    elif λ_precondition_pass_to_RW(simulation(), None):
        do Speak("Right Winger is available. Passing the ball to them.")
        do Pass(RightWinger)
        do Idle() for 2 seconds
        do Speak("Pass complete. I'm moving forward to support the Right Winger.")
        do MoveTo(λ_target_support_RW())
    else:
        do Speak("No good options. I'll hold the ball and wait for an opening.")
        do Idle()
    do Speak("I am in a good supporting position now.")
    do Idle()

A1precondition_pass_to_RS = HasPath({'obj1': 'Coach', 'obj2': 'RightStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LS = HasPath({'obj1': 'Coach', 'obj2': 'LeftStriker', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_LW = HasPath({'obj1': 'Coach', 'obj2': 'LeftWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1precondition_pass_to_RW = HasPath({'obj1': 'Coach', 'obj2': 'RightWinger', 'path_width': {'avg': 2.5, 'std': 0.5}})
A1target_support_RS = CloseTo({'obj': 'Coach', 'ref': 'RightStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LS = CloseTo({'obj': 'Coach', 'ref': 'LeftStriker', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_LW = CloseTo({'obj': 'Coach', 'ref': 'LeftWinger', 'max': {'avg': 6.0, 'std': 1.0}})
A1target_support_RW = CloseTo({'obj': 'Coach', 'ref': 'RightWinger', 'max': {'avg': 6.0, 'std': 1.0}})

def λ_precondition_pass_to_RS(scene, sample):
    return A1precondition_pass_to_RS.bool(simulation())

def λ_precondition_pass_to_LS(scene, sample):
    return A1precondition_pass_to_LS.bool(simulation())

def λ_precondition_pass_to_LW(scene, sample):
    return A1precondition_pass_to_LW.bool(simulation())

def λ_precondition_pass_to_RW(scene, sample):
    return A1precondition_pass_to_RW.bool(simulation())

def λ_target_support_RS():
    return A1target_support_RS.dist(simulation(), ego=True)

def λ_target_support_LS():
    return A1target_support_LS.dist(simulation(), ego=True)

def λ_target_support_LW():
    return A1target_support_LW.dist(simulation(), ego=True)

def λ_target_support_RW():
    return A1target_support_RW.dist(simulation(), ego=True)

def λ_termination(scene, sample):
    return False



# Ego (center midfielder) at origin
pi = 3.1415
ego = new Coach at (0, 0, 0), facing toward (0, 0, 0), with team "blue", with behavior CoachBehavior()

# Wingers
left_winger_angle = 90 + Uniform(0, 10)  # degrees from y-axis, 90 is positive x-axis (left), variance +/-10
right_winger_angle = -90 + Uniform(0, 10)  # degrees from y-axis, -90 is negative x-axis (right), variance +/-10
winger_dist = Uniform(6,8)

left_winger_x = winger_dist * sin(left_winger_angle * pi / 180)
left_winger_y = winger_dist * cos(left_winger_angle * pi / 180)
LeftWinger = new Player at (left_winger_x, left_winger_y, 0), facing toward ego, with name "LeftWinger", with team "blue"

right_winger_x = winger_dist * sin(right_winger_angle * pi / 180)
right_winger_y = winger_dist * cos(right_winger_angle * pi / 180)
RightWinger = new Player at (right_winger_x, right_winger_y, 0), facing toward ego, with name "RightWinger", with team "blue"

# Strikers
left_striker_angle = -Uniform(8, 20)
right_striker_angle = Uniform(8, 20)
striker_dist = Uniform(8,10)

left_striker_x = striker_dist * sin(left_striker_angle * pi / 180)
left_striker_y = striker_dist * cos(left_striker_angle * pi / 180)
LeftStriker = new Player at (left_striker_x, left_striker_y, 0), facing toward ego, with name "LeftStriker", with team "blue"

right_striker_x = striker_dist * sin(right_striker_angle * pi / 180)
right_striker_y = striker_dist * cos(right_striker_angle * pi / 180)
RightStriker = new Player at (right_striker_x, right_striker_y, 0), facing toward ego, with name "RightStriker", with team "blue"

# Ball at ego's feet
ball = new Ball at (0, 1, 0)

# Defenders: each assigned to one attacker, at a distance and angle in front of them, facing ego
# Helper function for defender placement
# (Scenic doesn't support functions in .scenic, so we inline the logic)

defender1_angle = Uniform(-10, 10)
defender1_dist = Uniform(2,4)
defender1_x = ego.position.x + defender1_dist * sin(defender1_angle * pi / 180)
defender1_y = ego.position.y + defender1_dist * cos(defender1_angle * pi / 180)
defender1 = new Player at (defender1_x, defender1_y, 0), facing toward ego, with team "red", with name "Defender1"

defender2_angle = Uniform(-30, 30)
defender2_dist = Uniform(1,2)
defender2_x = LeftWinger.position.x + defender2_dist * sin(defender2_angle * pi / 180)
defender2_y = LeftWinger.position.y + defender2_dist * cos(defender2_angle * pi / 180)
defender2 = new Player at (defender2_x, defender2_y, 0), facing toward ego, with team "red", with name "Defender2"

defender3_angle = Uniform(-30, 30)
defender3_dist = Uniform(1,2)
defender3_x = RightWinger.position.x + defender3_dist * sin(defender3_angle * pi / 180)
defender3_y = RightWinger.position.y + defender3_dist * cos(defender3_angle * pi / 180)
defender3 = new Player at (defender3_x, defender3_y, 0), facing toward ego, with team "red", with name "Defender3"

defender4_angle = Uniform(-30, 30)
defender4_dist = Uniform(1,2)
defender4_x = LeftStriker.position.x + defender4_dist * sin(defender4_angle * pi / 180)
defender4_y = LeftStriker.position.y + defender4_dist * cos(defender4_angle * pi / 180)
defender4 = new Player at (defender4_x, defender4_y, 0), facing toward ego, with team "red", with name "Defender4"

defender5_angle = Uniform(-30, 30)
defender5_dist = Uniform(1,2)
defender5_x = RightStriker.position.x + defender5_dist * sin(defender5_angle * pi / 180)
defender5_y = RightStriker.position.y + defender5_dist * cos(defender5_angle * pi / 180)
defender5 = new Player at (defender5_x, defender5_y, 0), facing toward ego, with team "red", with name "Defender5"

'''

In [102]:
from fix_gemini_scenic import Fix_Gemini_Scenic
from scenic_fc.api import api
fixer = Fix_Gemini_Scenic(code, feedback, url, api, context, synth_demo, demos)
fixed_code_distribute_delete_node, response_distribute_delete_node = fixer.run()

In [103]:
response_distribute_delete_node

'Based on the coach\'s feedback, I will adjust the `CoachBehavior` to modify the agent\'s actions after passing to the strikers.\n\n- **Feedback Analysis**: The coach specified that after passing to the Right Striker or Left Striker, the agent should not move up the field. The reason is tactical: moving forward would allow the defender marking the coach to follow easily. The correct action is to hold the position.\n\n- **Code Changes**: I will modify the two conditional blocks in the `CoachBehavior` function that handle passes to the `RightStriker` and `LeftStriker`.\n    - In both the `if λ_precondition_pass_to_RS(...)` block and the `elif λ_precondition_pass_to_LS(...)` block, I will replace the `do MoveTo(...)` action with `do Idle()`.\n    - I will also update the preceding `do Speak(...)` action in both blocks to reflect this change, so the narration accurately describes that the coach is holding their position instead of moving to support.\n    - The behavior for passing to the w

In [104]:
fixed_code_distribute_delete_node

'python\nbehavior CoachBehavior():\n    do Speak("I\'m waiting for the pass, ready to receive the ball.")\n    #do ReceiveBall()\n    do GetBallPossession(ball)\n    do Speak("I have possession. Now I\'m looking for the best passing option.")\n    if λ_precondition_pass_to_RS(simulation(), None):\n        do Speak("Right Striker is open. I\'m passing the ball to them now.")\n        do Pass(RightStriker)\n        do Idle() for 2 seconds\n        # CHANGE: Per coach feedback, the coach should not move upfield after passing to a striker.\n        # This prevents the defender from easily following. Replaced MoveTo with Idle.\n        do Speak("Pass complete. I will hold my position.")\n        do Idle()\n    elif λ_precondition_pass_to_LS(simulation(), None):\n        do Speak("Left Striker has space. I\'m sending the ball to them.")\n        do Pass(LeftStriker)\n        do Idle() for 2 seconds\n        # CHANGE: Per coach feedback, the coach should not move upfield after passing to a st